# Notebook 02 — Hybrid Search & Reciprocal Rank Fusion

## Objectives
- Understand BM25 keyword search and its scoring formula
- Build semantic (dense) search with cosine similarity
- Merge BM25 + dense results using Reciprocal Rank Fusion (RRF)
- Understand LangChain's EnsembleRetriever concept
- Apply mock Cohere reranking to refine results
- Use `day5.hybrid_search` for production-ready hybrid retrieval

## 1. BM25 keyword search with rank_bm25

In [ ]:
from rank_bm25 import BM25Okapi

docs = [
    "LangGraph builds stateful agent workflows",
    "BM25 is a keyword ranking algorithm",
    "FastAPI creates REST API endpoints",
    "Hybrid search combines BM25 and semantic methods",
]
tokenized = [d.lower().split() for d in docs]
bm25 = BM25Okapi(tokenized)

query = "BM25 ranking"
scores = bm25.get_scores(query.lower().split())
print("BM25 scores:")
for doc, score in zip(docs, scores):
    print(f"  {score:.3f}: {doc}")

## BM25 Formula

BM25 scores document `D` for query `Q` as:

$$\text{BM25}(D, Q) = \sum_{q \in Q} \text{IDF}(q) \cdot \frac{f(q,D) \cdot (k_1 + 1)}{f(q,D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

Where:
- `f(q, D)` = term frequency of query word `q` in document `D`
- `|D|` = document length; `avgdl` = average document length
- `k1 = 1.5`, `b = 0.75` are tuning constants
- `IDF(q) = log((N - df + 0.5) / (df + 0.5))` where `df` = document frequency

**Key insight**: BM25 penalizes very long documents and rewards rare terms — it's much better than TF-IDF for retrieval.

## 2. Reciprocal Rank Fusion (RRF)

In [ ]:
def rrf(ranked_lists, k=60):
    """RRF: merge multiple ranked lists. k=60 is the standard constant (Cormack 2009)."""
    scores = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# Demo: BM25 says [2,0,1], Dense says [0,2,3]
bm25_ranking  = [2, 0, 1]
dense_ranking = [0, 2, 3]
fused = rrf([bm25_ranking, dense_ranking])
print("RRF fused ranking:")
for doc_id, score in fused:
    print(f"  doc_{doc_id}: {score:.4f}")

print()
print("Why doc_0 and doc_2 rank highest: they appear in BOTH lists.")
print("doc_0 score =", round(1/(60+2) + 1/(60+1), 4), "(rank 1 in bm25, rank 0 in dense)")

## 3. EnsembleRetriever concept (LangChain)

LangChain's `EnsembleRetriever` wraps multiple retrievers and fuses them:

```python
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS

bm25_retriever = BM25Retriever.from_documents(docs)
dense_retriever = FAISS.from_documents(docs, embeddings).as_retriever()

ensemble = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5]  # equally weighted
)
results = ensemble.invoke("hybrid search BM25")
```

Our `HybridSearcher` in `day5.hybrid_search` implements the same concept without requiring external vector stores.

## 4. Query rewriting strategies

In [ ]:
def query_rewrite_inline(query: str, strategy: str = "expand") -> str:
    if strategy == "expand":
        return query + " detailed explanation overview"
    elif strategy == "hypothetical":
        return f"A document that answers '{query}' would say:"
    elif strategy == "decompose":
        parts = query.split(" and ")
        return parts[0].strip() if len(parts) > 1 else query
    return query

q = "What is BM25 and how does it differ from TF-IDF?"
print("Original   :", q)
print("Expand     :", query_rewrite_inline(q, "expand"))
print("Hypothetical:", query_rewrite_inline(q, "hypothetical"))
print("Decompose  :", query_rewrite_inline(q, "decompose"))

## 5. Cohere rerank (mock)

In [ ]:
# Real: import cohere; co.rerank(query=query, documents=docs, model="rerank-english-v3.0")
def mock_cohere_rerank(query, docs):
    """Mock rerank: sort by Jaccard-like word overlap with query."""
    query_words = set(query.lower().split())
    def overlap(doc):
        doc_words = set(doc.lower().split())
        return len(query_words & doc_words) / max(len(query_words | doc_words), 1)
    return sorted(docs, key=overlap, reverse=True)

sample_docs = [
    "Python is a general purpose programming language",
    "BM25 is a keyword ranking algorithm for search engines",
    "Docker containers package apps with all dependencies",
    "BM25 and TF-IDF are both ranking algorithms",
]
reranked = mock_cohere_rerank("BM25 ranking algorithm", sample_docs)
print("After rerank:")
for d in reranked:
    print(f"  {d}")

## 6. Setup sys.path

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

## 7. Using day5.hybrid_search

In [ ]:
from day5.hybrid_search import BM25Index, DenseIndex, HybridSearcher, reciprocal_rank_fusion

knowledge_base = [
    "LangGraph is a framework for building stateful multi-agent workflows",
    "BM25 is a ranking algorithm based on probabilistic retrieval framework",
    "FastAPI is a modern Python web framework for building REST APIs",
    "Hybrid search combines keyword and semantic retrieval methods",
    "RAGAS evaluates RAG systems using faithfulness and answer relevancy",
    "Docker containers package applications with all their dependencies",
    "LangSmith provides tracing and evaluation for LLM applications",
    "Reciprocal Rank Fusion merges multiple ranked lists into one",
]

searcher = HybridSearcher(knowledge_base)
results = searcher.search("BM25 ranking algorithm", top_k=3)
print("Hybrid search results:")
for r in results:
    print(f"  {r}")

## 8. Explain scores: BM25 vs Dense vs RRF

In [ ]:
explained = searcher.explain("hybrid search retrieval", top_k=4)
print(f"{'Doc':<65} {'RRF':>6} {'BM25':>6} {'Dense':>6}")
print("-" * 90)
for row in explained:
    print(f"{row['doc']:<65} {row['rrf_score']:>6.4f} {row['bm25_score']:>6.4f} {row['dense_score']:>6.4f}")

## 9. Compare BM25-only vs Dense-only vs Hybrid

In [ ]:
from day5.hybrid_search import BM25Index, DenseIndex

query = "LangGraph multi-agent workflow"
bm25_only  = BM25Index(knowledge_base).get_top_docs(query, top_k=3)
dense_only = DenseIndex(knowledge_base).get_top_docs(query, top_k=3)
hybrid     = searcher.search(query, top_k=3)

print("BM25 only:")
for d in bm25_only:  print(f"  {d}")

print("\nDense only:")
for d in dense_only: print(f"  {d}")

print("\nHybrid (RRF):")
for d in hybrid:     print(f"  {d}")